In [ ]:
import plotly.express as px
import numpy as np
import pandas as pd

# 1. Carga de datos y filtrado
df = pd.read_parquet('../data/casen_2024.parquet')

# Diccionarios de mapeo
educc_map = {
    0: 'Sin Educación<br>Formal', 1: 'Básica<br>Incompleta', 2: 'Básica<br>Completa',
    3: 'Media<br>Incompleta', 4: 'Media<br>Completa', 5: 'Superior<br>Incompleta', 6: 'Superior<br>Completa'
}
# 'area' en CASEN: 1 es Urbano, 2 es Rural
zona_map = {
    1: 'Urbano', 
    2: 'Rural'
}

# Filtramos valores desconocidos y copiamos
df_educ = df[(df['educc'] >= 0) & (df['area'].isin([1, 2]))].copy()

# Mapeamos las categorías a texto
df_educ['Nivel Educacional'] = df_educ['educc'].map(educc_map)
df_educ['Zona'] = df_educ['area'].map(zona_map)

# Agrupamos contando las personas por ambas categorías conjuntas
educ_zona_counts = df_educ.groupby(['Zona', 'Nivel Educacional']).size().reset_index(name='Cantidad')

# Creamos el TreeMap interactivo con 2 niveles
fig = px.treemap(
    educ_zona_counts, 
    path=['Zona', 'Nivel Educacional'], # Jerarquía: Primero Zona, luego Nivel Educacional
    values='Cantidad',
    title='TreeMap: Proporción por Zona y Nivel Educacional',
    color='Cantidad',
    color_continuous_scale='Blues'
)

# Generamos marcas dinámicas para la leyenda (ej. 10 mil, 20 mil)
max_val = educ_zona_counts['Cantidad'].max()
step = 10000
tickvals = np.arange(0, max_val + step, step)
ticktext = [f"{int(v/1000)} mil" if v > 0 else "0" for v in tickvals]

fig.update_layout(
    font=dict(size=18),
    margin=dict(t=50, l=50, r=25, b=25),
    coloraxis_colorbar=dict(
        title="Cantidad",
        tickvals=tickvals,
        ticktext=ticktext
    )
)

fig.update_traces(textfont=dict(size=22), hoverlabel=dict(font_size=16))
fig.show()